In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

from xgboost import XGBRegressor

In [2]:
df = pd.read_csv("../data/sales_data.csv")

df.head()

,State,Date,Total,Category
0,Alabama,1/12/2019,"109,574,036",Beverages
1,Arizona,1/12/2019,"109,101,595",Beverages
2,Arkansas,1/12/2019,"58,049,432",Beverages
3,California,1/12/2019,"444,766,891",Beverages
4,Colorado,1/12/2019,"89,816,716",Beverages


In [3]:
df["Date"] = pd.to_datetime(df["Date"], format="mixed")

df["Total"] = df["Total"].str.replace(",", "").astype(int)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8084 entries, 0 to 8083
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   State     8084 non-null   str           
 1   Date      8084 non-null   datetime64[us]
 2   Total     8084 non-null   int64         
 3   Category  8084 non-null   str           
dtypes: datetime64[us](1), int64(1), str(2)
memory usage: 252.8 KB


In [4]:
df.head()

,State,Date,Total,Category
0,Alabama,2019-01-12,109574036,Beverages
1,Arizona,2019-01-12,109101595,Beverages
2,Arkansas,2019-01-12,58049432,Beverages
3,California,2019-01-12,444766891,Beverages
4,Colorado,2019-01-12,89816716,Beverages


In [5]:
df = df.drop("Category", axis=1)

df.head()

,State,Date,Total
0,Alabama,2019-01-12,109574036
1,Arizona,2019-01-12,109101595
2,Arkansas,2019-01-12,58049432
3,California,2019-01-12,444766891
4,Colorado,2019-01-12,89816716


In [7]:
df = df.sort_values(["State", "Date"])

df.head(10)

,State,Date,Total
0,Alabama,2019-01-12,109574036
43,Alabama,2019-03-11,112189104
86,Alabama,2019-06-10,129106730
129,Alabama,2019-08-12,108083724
172,Alabama,2019-10-11,110932913
3397,Alabama,2019-10-13,123782286
5246,Alabama,2019-10-20,116218909
7095,Alabama,2019-10-27,109968011
4472,Alabama,2019-11-17,109056410
6321,Alabama,2019-11-24,113040422


In [8]:
encoder = LabelEncoder()

df["state_encoded"] = encoder.fit_transform(df["State"])

df.head()

,State,Date,Total,state_encoded
0,Alabama,2019-01-12,109574036,0
43,Alabama,2019-03-11,112189104,0
86,Alabama,2019-06-10,129106730,0
129,Alabama,2019-08-12,108083724,0
172,Alabama,2019-10-11,110932913,0


In [9]:
df["month"] = df["Date"].dt.month

df["quarter"] = df["Date"].dt.quarter

df["week_of_year"] = df["Date"].dt.isocalendar().week

In [10]:
df["lag_1"] = df.groupby("State")["Total"].shift(1)

df["lag_7"] = df.groupby("State")["Total"].shift(7)

df["lag_30"] = df.groupby("State")["Total"].shift(30)

In [11]:
df.head(15)

,State,Date,Total,state_encoded,month,quarter,week_of_year,lag_1,lag_7,lag_30
0,Alabama,2019-01-12,109574036,0,1,1,2,NaN,NaN,NaN
43,Alabama,2019-03-11,112189104,0,3,1,11,109574036.0,NaN,NaN
86,Alabama,2019-06-10,129106730,0,6,2,24,112189104.0,NaN,NaN
129,Alabama,2019-08-12,108083724,0,8,3,33,129106730.0,NaN,NaN
172,Alabama,2019-10-11,110932913,0,10,4,41,108083724.0,NaN,NaN
3397,Alabama,2019-10-13,123782286,0,10,4,41,110932913.0,NaN,NaN
5246,Alabama,2019-10-20,116218909,0,10,4,42,123782286.0,NaN,NaN
7095,Alabama,2019-10-27,109968011,0,10,4,43,116218909.0,109574036.0,NaN
4472,Alabama,2019-11-17,109056410,0,11,4,46,109968011.0,112189104.0,NaN
6321,Alabama,2019-11-24,113040422,0,11,4,47,109056410.0,129106730.0,NaN


In [12]:
df["rolling_mean_4"] = (
    df.groupby("State")["Total"]
    .transform(lambda x: x.rolling(4).mean())
)

df["rolling_std_4"] = (
    df.groupby("State")["Total"]
    .transform(lambda x: x.rolling(4).std())
)


In [14]:
df.head(15)

,State,Date,Total,state_encoded,month,quarter,week_of_year,lag_1,lag_7,lag_30,rolling_mean_4,rolling_std_4
0,Alabama,2019-01-12,109574036,0,1,1,2,NaN,NaN,NaN,NaN,NaN
43,Alabama,2019-03-11,112189104,0,3,1,11,109574036.0,NaN,NaN,NaN,NaN
86,Alabama,2019-06-10,129106730,0,6,2,24,112189104.0,NaN,NaN,NaN,NaN
129,Alabama,2019-08-12,108083724,0,8,3,33,129106730.0,NaN,NaN,1.147384e+08,9.728021e+06
172,Alabama,2019-10-11,110932913,0,10,4,41,108083724.0,NaN,NaN,1.150781e+08,9.508814e+06
3397,Alabama,2019-10-13,123782286,0,10,4,41,110932913.0,NaN,NaN,1.179764e+08,1.008412e+07
5246,Alabama,2019-10-20,116218909,0,10,4,42,123782286.0,NaN,NaN,1.147545e+08,6.898048e+06
7095,Alabama,2019-10-27,109968011,0,10,4,43,116218909.0,109574036.0,NaN,1.152255e+08,6.331744e+06
4472,Alabama,2019-11-17,109056410,0,11,4,46,109968011.0,112189104.0,NaN,1.147564e+08,6.807449e+06
6321,Alabama,2019-11-24,113040422,0,11,4,47,109056410.0,129106730.0,NaN,1.120709e+08,3.248345e+06


In [15]:
df = df.dropna()

df.shape

(6794, 12)

In [16]:
X = df[[
    "state_encoded",
    "lag_1",
    "lag_7",
    "lag_30",
    "rolling_mean_4",
    "rolling_std_4",
    "month",
    "quarter",
    "week_of_year"
]]

y = df["Total"]

In [17]:
X.head()

,state_encoded,lag_1,lag_7,lag_30,rolling_mean_4,rolling_std_4,month,quarter,week_of_year
559,0,123259529.0,127145065.0,109574036.0,1.210072e+08,1.411384e+07,5,2,19
4343,0,138414683.0,116657084.0,112189104.0,1.240636e+08,1.459419e+07,5,2,20
6192,0,130340051.0,133713630.0,129106730.0,1.316807e+08,6.512252e+06,5,2,21
7955,0,134708619.0,117742073.0,108083724.0,1.333378e+08,4.022535e+06,5,2,22
602,0,129887877.0,118114595.0,110932913.0,1.332106e+08,3.810881e+06,6,2,24


In [18]:
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print(X_train.shape)
print(X_test.shape)

(5435, 9)
(1359, 9)


In [19]:
xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42
)

xgb_model.fit(X_train, y_train)

print("Professional XGBoost model trained successfully!")

Professional XGBoost model trained successfully!


In [20]:
y_pred = xgb_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)

rmse = root_mean_squared_error(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 9533646.0
RMSE: 21421764.0


In [21]:
import joblib

joblib.dump(xgb_model, "../models/xgboost_multistate.pkl")

joblib.dump(encoder, "../models/state_encoder.pkl")

print("Professional multi-state model saved!")

Professional multi-state model saved!
